# Notebook 03 — Model Training & AutoML

## Objectif

L'objectif de ce notebook est d'entraîner et de comparer plusieurs modèles de Machine Learning pour prédire le Churn des clients.

Les étapes :
1. Charger les données prétraitées.
2. Initialiser plusieurs algorithmes.
3. Entraîner les modèles.
4. Évaluer leurs performances.
5. Comparer les résultats.
6. Sélectionner automatiquement le meilleur modèle selon le Mean F1-score (Validation Croisée).
7. Tracer les courbes ROC et sauvegarder le graphique.
8. Optimiser les hyperparamètres (GridSearchCV) et sauvegarder le modèle final et les rapports.

## 1. Importation des bibliothèques

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.model_selection import cross_val_score, GridSearchCV

pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

print("Bibliothèques importées avec succès !")

## 2. Définition des chemins

In [ ]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("Projet :", PROJECT_ROOT)
print("Données :", PROCESSED_DIR)
print("Modèles :", MODELS_DIR)
print("Rapports :", REPORTS_DIR)

required_files = [
    PROCESSED_DIR / "X_train_processed.csv",
    PROCESSED_DIR / "X_test_processed.csv",
    PROCESSED_DIR / "y_train.csv",
    PROCESSED_DIR / "y_test.csv"
]

missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        "Fichiers manquants. Exécutez d'abord le Notebook 02 :\n"
        + "\n".join(missing_files)
    )

## 3. Chargement des données prétraitées

In [ ]:
X_train = pd.read_csv(PROCESSED_DIR / "X_train_processed.csv")
X_test = pd.read_csv(PROCESSED_DIR / "X_test_processed.csv")

y_train = pd.read_csv(PROCESSED_DIR / "y_train.csv")["Churn"]
y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv")["Churn"]

print("Dimensions des données :")
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

display(X_train.head())

## 4. Vérification des données

Nous vérifions les valeurs manquantes, les dimensions et les classes de la cible avant l'entraînement.

In [ ]:
print("Valeurs manquantes dans X_train :", X_train.isnull().sum().sum())
print("Valeurs manquantes dans X_test  :", X_test.isnull().sum().sum())

print("\nDistribution de Churn dans Train :")
display(y_train.value_counts().sort_index())

print("\nDistribution de Churn dans Test :")
display(y_test.value_counts().sort_index())

print("\nValeurs de la cible :", sorted(y_train.unique()))

assert X_train.columns.equals(X_test.columns), "Les colonnes Train et Test ne correspondent pas."
assert set(y_train.unique()).issubset({0, 1}), "La cible Train doit contenir 0 et 1."
assert set(y_test.unique()).issubset({0, 1}), "La cible Test doit contenir 0 et 1."

## 5. Initialisation des modèles

Nous allons comparer six algorithmes :
- **Logistic Regression** : modèle de classification de référence.
- **Decision Tree** : prend des décisions sous forme d'arbre.
- **Random Forest** : combine plusieurs arbres de décision.
- **KNN** : classe un client selon ses voisins les plus proches.
- **SVM** : cherche une frontière séparant les classes (calibré avec `CalibratedClassifierCV`).
- **Gradient Boosting** : construit des modèles successifs pour corriger les erreurs précédentes.

Le paramètre `class_weight="balanced"` est utilisé dans certains modèles pour tenir compte du déséquilibre entre les classes.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),
    "SVM": CalibratedClassifierCV(
        SVC(class_weight="balanced", random_state=42)
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    )
}

print("Nombre de modèles :", len(models))
for name in models:
    print("-", name)

## 6. Entraînement des modèles

In [ ]:
trained_models = {}

for name, model in models.items():
    print(f"Entraînement de {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model

print("\nTous les modèles ont été entraînés avec succès !")

## 7. Évaluation des modèles

Nous calculons cinq métriques :
- **Accuracy** : proportion de prédictions correctes.
- **Precision** : parmi les clients prédits comme ayant quitté l'entreprise, proportion réellement partie.
- **Recall** : parmi les clients réellement partis, proportion détectée.
- **F1-score** : équilibre entre Precision et Recall.
- **ROC-AUC** : capacité à distinguer les deux classes.

Dans un problème de Churn, il est important d'examiner le Recall et le F1-score, et pas seulement l'Accuracy.

In [ ]:
results = []

for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        by="F1-score",
        ascending=False
    ).reset_index(drop=True).round(4)
)

## 8. Comparaison graphique des modèles

In [ ]:
plt.figure(figsize=(11, 6))

sns.barplot(
    data=results_df.sort_values("F1-score", ascending=False),
    x="F1-score",
    y="Model"
)

plt.title("Comparaison des modèles selon le F1-score")
plt.xlabel("F1-score")
plt.ylabel("Modèle")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## 9. Validation croisée (Cross Validation)

La validation croisée à 5 folds divise les données d'entraînement en cinq parties. Chaque partie est utilisée une fois pour valider le modèle, tandis que les autres servent à l'entraînement.

Cela permet d'observer si les performances sont relativement stables selon les divisions des données.

In [ ]:
cv_results = []

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="f1",
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "Mean F1-score": scores.mean(),
        "Std F1-score": scores.std()
    })

cv_df = pd.DataFrame(cv_results)

display(
    cv_df.sort_values(
        by="Mean F1-score",
        ascending=False
    ).reset_index(drop=True).round(4)
)

## 10. Sélection automatique du meilleur modèle

Nous sélectionnons automatiquement le meilleur modèle selon le **Mean F1-score** obtenu en validation croisée (Cross-Validation à 5 folds) afin de garantir un choix robuste et d'éviter le surapprentissage sur le jeu de test.

In [ ]:
best_model_name = cv_df.loc[
    cv_df["Mean F1-score"].idxmax(),
    "Model"
]

best_model = trained_models[best_model_name]

best_cv_f1 = cv_df.loc[
    cv_df["Model"] == best_model_name,
    "Mean F1-score"
].iloc[0]

best_test_f1 = results_df.loc[
    results_df["Model"] == best_model_name,
    "F1-score"
].iloc[0]

print("Meilleur modèle (selon Mean F1-score CV) :", best_model_name)
print(f"Mean F1-score (CV) : {best_cv_f1:.4f}")
print(f"F1-score (Test)    : {best_test_f1:.4f}")

## 11. Rapport de classification

In [ ]:
y_best_pred = best_model.predict(X_test)

print("Classification Report :\n")
print(
    classification_report(
        y_test,
        y_best_pred,
        target_names=["No Churn", "Churn"],
        zero_division=0
    )
)

## 12. Matrice de confusion

La matrice de confusion permet d'analyser les erreurs :
- **True Negative** : le client reste et le modèle prédit `No Churn`.
- **True Positive** : le client part et le modèle prédit `Churn`.
- **False Positive** : le modèle prédit un départ alors que le client reste.
- **False Negative** : le client part, mais le modèle ne détecte pas son départ.

In [ ]:
cm = confusion_matrix(y_test, y_best_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Churn", "Churn"]
)

disp.plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()

## 13. Courbes ROC

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for name, model in trained_models.items():
    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        name=name,
        ax=ax
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aléatoire (AUC = 0.50)")
ax.set_title("Courbes ROC — Comparaison des modèles")
ax.set_xlabel("Taux de Faux Positifs (FPR)")
ax.set_ylabel("Taux de Vrais Positifs (TPR)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "roc_curves.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Courbes ROC sauvegardées dans : {REPORTS_DIR / 'roc_curves.png'}")

## 14. Optimisation des hyperparamètres (GridSearchCV) et sauvegarde

In [ ]:
param_grids = {
    "Logistic Regression": {
        "C": [0.01, 0.1, 1.0, 10.0],
        "solver": ["lbfgs", "liblinear"]
    },
    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20]
    },
    "Gradient Boosting": {
        "n_estimators": [100, 200],
        "learning_rate": [0.05, 0.1]
    }
}

base_estimator = models[best_model_name]
param_grid = param_grids.get(best_model_name, {})

grid_search = GridSearchCV(
    estimator=base_estimator,
    param_grid=param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
final_model = grid_search.best_estimator_

y_final_pred = final_model.predict(X_test)
y_final_proba = final_model.predict_proba(X_test)[:, 1]

final_metrics = {
    "best_model": best_model_name,
    "best_params": grid_search.best_params_,
    "best_cv_f1": round(float(grid_search.best_score_), 4),
    "test_accuracy": round(float(accuracy_score(y_test, y_final_pred)), 4),
    "test_precision": round(float(precision_score(y_test, y_final_pred, zero_division=0)), 4),
    "test_recall": round(float(recall_score(y_test, y_final_pred, zero_division=0)), 4),
    "test_f1_score": round(float(f1_score(y_test, y_final_pred, zero_division=0)), 4),
    "test_roc_auc": round(float(roc_auc_score(y_test, y_final_proba)), 4)
}

print("Meilleurs hyperparamètres :", grid_search.best_params_)
print(f"Meilleur Mean F1-score (CV) : {grid_search.best_score_:.4f}")
print(f"F1-score final (Test)       : {final_metrics['test_f1_score']:.4f}")

# Sauvegarde du modèle optimisé
model_path = MODELS_DIR / "best_model.joblib"
joblib.dump(final_model, model_path)
print(f"Modèle sauvegardé dans : {model_path}")

# Sauvegarde des rapports et métriques
results_df.to_csv(REPORTS_DIR / "model_comparison_test.csv", index=False)
cv_df.to_csv(REPORTS_DIR / "model_comparison_cv.csv", index=False)

metrics_path = REPORTS_DIR / "best_model_metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=4, ensure_ascii=False)

print(f"Rapports sauvegardés dans : {REPORTS_DIR}")